# Logistic Regression - Spam Filter Example 📧

## 🎯 Goal
Build a **spam email filter** using Logistic Regression!

## 📚 What You'll Learn
- The sigmoid function and probability
- How to train a Logistic Regression model on email data
- How to interpret coefficients
- Decision boundaries (where do we draw the line?)
- All evaluation metrics (accuracy, precision, recall, F1, ROC-AUC)
- Why **PRECISION** is critical for spam filters
- How to tune the threshold for fewer false alarms

## Step 1: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc
)

np.random.seed(42)
sns.set_style('whitegrid')

print('✅ Libraries imported!')

## Step 2: Visualize the Sigmoid Function

First, let's understand the magic behind logistic regression!

In [ ]:
def sigmoid(z):
    """The sigmoid function - converts any number to 0-1"""
    return 1 / (1 + np.exp(-z))

# Test sigmoid with different values
test_values = [-5, -2, -1, 0, 1, 2, 5]
print('Sigmoid Examples:')
print('=' * 40)
for z in test_values:
    print(f'  σ({z:3d}) = {sigmoid(z):.4f} ({sigmoid(z)*100:.1f}%)')

# Plot the sigmoid
z = np.linspace(-10, 10, 100)
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(z, sigmoid(z), linewidth=3, color='blue')
ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Threshold (0.5)')
ax.axvline(x=0, color='green', linestyle='--', alpha=0.5, label='z = 0')
ax.scatter([0], [0.5], color='red', s=100, zorder=5)
ax.set_xlabel('Input (z)', fontsize=12)
ax.set_ylabel('Output σ(z) = P(spam)', fontsize=12)
ax.set_title('The Sigmoid Function: Squishes Any Number to 0-1', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 3: Create Email Dataset

Generate realistic email data with these features:
- **suspicious_words**: count of words like 'free', 'win', 'click', 'urgent', 'offer'
- **caps_ratio**: percentage of ALL CAPS characters in subject

In [ ]:
# Generate email data
n_emails = 300

# Features
suspicious_words = np.random.uniform(0, 15, n_emails)
caps_ratio = np.random.uniform(0, 100, n_emails)

# True relationship (with noise)
# More spam-like words and ALL CAPS = more likely spam
score = -5 + 0.6*suspicious_words + 0.05*caps_ratio + np.random.normal(0, 1, n_emails)
is_spam = (score > 0).astype(int)

df = pd.DataFrame({
    'suspicious_words': suspicious_words.round(1),
    'caps_ratio': caps_ratio.round(1),
    'is_spam': is_spam
})

print('📧 Email Dataset:')
print(df.head(10))
print(f'\nTotal emails: {len(df)}')
print(f'Spam:     {df["is_spam"].sum()} ({df["is_spam"].mean()*100:.1f}%)')
print(f'Not spam: {(1-df["is_spam"]).sum()} ({(1-df["is_spam"]).mean()*100:.1f}%)')

print('\n💡 Example emails:')
print('  Low score:   "Hi mom, are we meeting tomorrow?"        → Inbox 📬')
print('  High score:  "FREE!!! WIN $1000 NOW! CLICK HERE!!!"   → Spam 🚫')

## Step 4: Visualize the Email Data

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

# Plot non-spam emails
ax.scatter(df[df['is_spam']==0]['suspicious_words'], df[df['is_spam']==0]['caps_ratio'],
           c='green', label='📬 Not Spam (Inbox)', s=80, edgecolors='black', alpha=0.7)

# Plot spam emails
ax.scatter(df[df['is_spam']==1]['suspicious_words'], df[df['is_spam']==1]['caps_ratio'],
           c='red', label='🚫 Spam', s=80, edgecolors='black', alpha=0.7, marker='X')

ax.set_xlabel('Suspicious Words Count', fontsize=12)
ax.set_ylabel('ALL CAPS Percentage', fontsize=12)
ax.set_title('Email Classification: Spam vs Not Spam', fontsize=14, fontweight='bold')
ax.legend(fontsize=12, loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('\n💡 Notice the pattern:')
print('   More suspicious words + more CAPS = more likely SPAM')
print('   We can draw a line to separate them!')

## Step 5: Prepare Data and Train Model

In [ ]:
# Split data
X = df[['suspicious_words', 'caps_ratio']]
y = df['is_spam']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features (good practice!)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train model
model = LogisticRegression()
model.fit(X_train_scaled, y_train)

print('🤖 Spam Filter trained!')
print(f'\n📋 Model Parameters:')
print(f'  Intercept (β₀):       {model.intercept_[0]:.4f}')
print(f'  Suspicious words coef: {model.coef_[0][0]:.4f}')
print(f'  Caps ratio coef:       {model.coef_[0][1]:.4f}')

print('\n💡 Interpretation:')
print('  - Both coefficients are POSITIVE')
print('  - More suspicious words → higher P(spam) 🚫')
print('  - More ALL CAPS → higher P(spam) 🚫')

## Step 6: Make Predictions on Real-Looking Emails

In [ ]:
# Get predictions and probabilities on test set
predictions = model.predict(X_test_scaled)
probabilities = model.predict_proba(X_test_scaled)

# Show first 10 predictions
results_df = pd.DataFrame({
    'Suspicious Words': X_test['suspicious_words'].values[:10],
    'Caps %': X_test['caps_ratio'].values[:10],
    'P(Inbox)': probabilities[:10, 0].round(3),
    'P(Spam)': probabilities[:10, 1].round(3),
    'Predicted': ['🚫 Spam' if p == 1 else '📬 Inbox' for p in predictions[:10]],
    'Actual': ['🚫 Spam' if a == 1 else '📬 Inbox' for a in y_test.values[:10]]
})

results_df['Correct'] = results_df['Predicted'] == results_df['Actual']
print('🔮 First 10 Email Predictions:')
print(results_df.to_string(index=False))

## Step 7: Evaluate the Spam Filter

In [ ]:
# Calculate all metrics
accuracy = accuracy_score(y_test, predictions)
precision = precision_score(y_test, predictions)
recall = recall_score(y_test, predictions)
f1 = f1_score(y_test, predictions)

print('📊 Spam Filter Performance:')
print('=' * 60)
print(f'  Accuracy:  {accuracy:.2%}')
print(f'  Precision: {precision:.2%}  ← Of flagged spam, how many were really spam?')
print(f'  Recall:    {recall:.2%}  ← Of actual spam, how many did we catch?')
print(f'  F1 Score:  {f1:.2%}')

print('\n📋 Full Classification Report:')
print(classification_report(y_test, predictions, target_names=['Not Spam (Inbox)', 'Spam']))

print('\n💡 For SPAM FILTERS, PRECISION is critical!')
print('   - High precision = few legitimate emails flagged as spam ✅')
print('   - Low precision = boss\'s email goes to spam folder! 😱')

## Step 8: Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, predictions)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['📬 Inbox', '🚫 Spam'],
            yticklabels=['📬 Inbox', '🚫 Spam'],
            cbar_kws={'label': 'Count'},
            ax=ax, annot_kws={'fontsize': 16})
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
ax.set_title('Spam Filter Confusion Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

TN, FP = cm[0]
FN, TP = cm[1]

print(f'\n📊 Breakdown:')
print(f'  ✅ True Negatives:  {TN}  (correctly kept in inbox)')
print(f'  ❌ False Positives: {FP}  (legitimate email marked as spam - BAD!)')
print(f'  ❌ False Negatives: {FN}  (spam slipped into inbox)')
print(f'  ✅ True Positives:  {TP}  (correctly caught spam)')

if FP > 0:
    print(f'\n   ⚠️ {FP} legitimate emails went to spam! Consider higher threshold.')

## Step 9: Visualize the Decision Boundary

See where the model draws the line between Spam and Inbox!

In [ ]:
# Create mesh
x_min, x_max = X['suspicious_words'].min() - 1, X['suspicious_words'].max() + 1
y_min, y_max = X['caps_ratio'].min() - 5, X['caps_ratio'].max() + 5

xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 200),
    np.linspace(y_min, y_max, 200)
)

# Scale and predict
grid = np.c_[xx.ravel(), yy.ravel()]
grid_scaled = scaler.transform(grid)
Z = model.predict_proba(grid_scaled)[:, 1].reshape(xx.shape)

# Plot
fig, ax = plt.subplots(figsize=(12, 7))

# Probability contour (red = spam zone)
contour = ax.contourf(xx, yy, Z, levels=20, cmap='RdYlGn_r', alpha=0.5)
plt.colorbar(contour, label='P(Spam)')

# Decision boundary at 0.5
ax.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=3)

# Data points
ax.scatter(df[df['is_spam']==0]['suspicious_words'], df[df['is_spam']==0]['caps_ratio'],
           c='green', label='📬 Not Spam (Inbox)', s=80, edgecolors='black', alpha=0.8)
ax.scatter(df[df['is_spam']==1]['suspicious_words'], df[df['is_spam']==1]['caps_ratio'],
           c='red', label='🚫 Spam', s=80, edgecolors='black', alpha=0.8, marker='X')

ax.set_xlabel('Suspicious Words Count', fontsize=12)
ax.set_ylabel('ALL CAPS %', fontsize=12)
ax.set_title('Spam Filter Decision Boundary', fontsize=14, fontweight='bold')
ax.legend(fontsize=12, loc='upper left')
plt.tight_layout()
plt.show()

print('\n💡 The black line is the decision boundary (P=0.5)')
print('   Right of the line → Spam | Left → Inbox')
print('   Color shows confidence (redder = more likely spam)')

## Step 10: ROC Curve and AUC

In [ ]:
# Calculate ROC curve
fpr, tpr, thresholds = roc_curve(y_test, probabilities[:, 1])
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(fpr, tpr, color='blue', linewidth=3, label=f'ROC (AUC = {roc_auc:.3f})')
ax.plot([0, 1], [0, 1], color='red', linestyle='--', label='Random (AUC = 0.5)')
ax.fill_between(fpr, tpr, alpha=0.2, color='blue')
ax.set_xlabel('False Positive Rate (Inbox emails marked spam)', fontsize=12)
ax.set_ylabel('True Positive Rate (Spam caught)', fontsize=12)
ax.set_title('Spam Filter ROC Curve', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'\n📊 AUC Score: {roc_auc:.3f}')
if roc_auc >= 0.9:
    print('   ✅ Excellent spam filter!')
elif roc_auc >= 0.8:
    print('   ✅ Good spam filter!')
elif roc_auc >= 0.7:
    print('   ✅ Decent spam filter')
else:
    print('   ⚠️ Needs improvement')

## Step 11: Tune Threshold for Better Spam Filter

**For spam filters, we want HIGH PRECISION** - don't block legitimate emails!
Let's try a higher threshold.

In [ ]:
thresholds_to_try = [0.3, 0.5, 0.7, 0.8, 0.9]
results = []

for thresh in thresholds_to_try:
    pred_thresh = (probabilities[:, 1] >= thresh).astype(int)
    cm_t = confusion_matrix(y_test, pred_thresh)
    fp = cm_t[0, 1]  # Legitimate emails marked as spam
    
    results.append({
        'Threshold': thresh,
        'Precision': precision_score(y_test, pred_thresh, zero_division=0),
        'Recall': recall_score(y_test, pred_thresh, zero_division=0),
        'F1': f1_score(y_test, pred_thresh, zero_division=0),
        'False Positives': fp,
        'Note': '⚠️ Risky' if fp > 2 else '✅ Safe'
    })

thresh_df = pd.DataFrame(results)
print('🎚️ Threshold Tuning for Spam Filter:')
print('=' * 70)
print(thresh_df.round(3).to_string(index=False))

print('\n💡 Key insight for spam filters:')
print('   - LOWER threshold (0.3) → catches more spam BUT blocks legitimate emails')
print('   - HIGHER threshold (0.8) → fewer false alarms, but some spam slips through')
print('   - Best for spam filter: 0.7-0.9 (prioritize precision!)')

## Step 12: Build a Spam Detector Function!

Use it to classify new incoming emails.

In [ ]:
def classify_email(suspicious_words, caps_ratio, threshold=0.5, email_preview=''):
    """Classify an email as spam or not"""
    features = np.array([[suspicious_words, caps_ratio]])
    features_scaled = scaler.transform(features)
    
    prob = model.predict_proba(features_scaled)[0, 1]
    is_spam_pred = prob >= threshold
    
    print(f'\n📧 Email: "{email_preview}"')
    print(f'   Suspicious words: {suspicious_words}')
    print(f'   ALL CAPS %: {caps_ratio}%')
    print(f'\n   📊 P(Spam) = {prob:.1%}')
    print(f'   🎯 Decision (threshold={threshold}): {"🚫 SPAM" if is_spam_pred else "📬 INBOX"}')
    
    if prob > 0.9:
        print(f'   💥 Very high confidence spam')
    elif prob > 0.6:
        print(f'   ⚠️ Likely spam')
    elif prob > 0.4:
        print(f'   🤔 Uncertain - borderline')
    elif prob > 0.1:
        print(f'   ✅ Probably legitimate')
    else:
        print(f'   ✅ Definitely legitimate')

# Test cases - simulating real emails
classify_email(suspicious_words=1, caps_ratio=5, 
               email_preview="Hi mom, are we still on for dinner?")

classify_email(suspicious_words=4, caps_ratio=15, 
               email_preview="Reminder: Project deadline next Friday")

classify_email(suspicious_words=8, caps_ratio=40, 
               email_preview="FREE trial offer - limited time!")

classify_email(suspicious_words=12, caps_ratio=80, 
               email_preview="WIN $1000 NOW!!! CLICK HERE!!! URGENT!!!")

# Use higher threshold for stricter filter
print('\n\n🔒 Using HIGHER threshold (0.8) for stricter filter:')
classify_email(suspicious_words=8, caps_ratio=40, threshold=0.8,
               email_preview="FREE trial offer - limited time!")

## 🎓 Summary

### What We Built:
1. ✅ Visualized the sigmoid function
2. ✅ Created realistic email dataset
3. ✅ Trained a Logistic Regression spam filter
4. ✅ Made predictions with confidence scores
5. ✅ Evaluated with multiple metrics
6. ✅ Visualized confusion matrix and decision boundary
7. ✅ Plotted ROC curve and calculated AUC
8. ✅ Tuned threshold to reduce false alarms
9. ✅ Built a working spam classifier function!

### Key Takeaways:
- **Sigmoid** converts any number to a probability (0-1)
- **Threshold of 0.5** is default but **higher is better for spam filters**
- **PRECISION matters most** for spam filters (don't block legitimate mail!)
- **Decision boundary** shows where model is uncertain
- **ROC-AUC** is a great metric to compare models

### Real-World Spam Filter Considerations:
1. **Thousands of features** (not just 2): word frequencies, sender reputation, links, attachments, time of day
2. **Imbalanced data**: usually <10% emails are spam
3. **Adversarial**: spammers actively try to evade detection
4. **Multi-language support**
5. **Use ensemble methods** (Random Forest, XGBoost) for better performance

### Try These Exercises:
1. Add more features (sender reputation, has_attachment, time_sent)
2. Try with imbalanced data (class_weight='balanced')
3. Add regularization (penalty='l1' or 'l2')
4. Use a real spam dataset (e.g., SpamAssassin, SMS Spam Collection)

**Happy Spam Hunting! 📧🎯✨**